# Análise exploratória — Titanic

Atividade da Aula 4 (Dados 1) da Trilha UFPE.

Escolhi o dataset do Titanic: 891 passageiros, com informações como classe, sexo, idade, tarifa e se a pessoa sobreviveu ou não. Ele tem valores faltantes e algumas colunas com tipo errado, então dá pra passar por todo o roteiro da aula.

Perguntas que quero responder:

1. Mulheres realmente sobreviveram mais que homens?
2. A classe do passageiro fez diferença na sobrevivência?
3. Crianças tiveram mais chance de sobreviver?

## 1. Carregar e inspecionar

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
df.shape

(891, 12)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [4]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
df.describe(exclude="number")

,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


In [6]:
for col in ["Sex", "Pclass", "Embarked"]:
    print(df[col].value_counts(dropna=False), "\n")

Sex
male      577
female    314
Name: count, dtype: int64 

Pclass
3    491
1    216
2    184
Name: count, dtype: int64 

Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64 



Cada linha é um passageiro. Algumas coisas que já chamam atenção:

- `Age`, `Cabin` e `Embarked` têm menos de 891 valores, ou seja, têm dados faltando.
- `Survived` está como número (0/1), mas na verdade é sim/não.
- `Pclass` também está como número, mas é uma categoria (1ª, 2ª e 3ª classe). Calcular a "média da classe" não faz muito sentido.
- A tarifa (`Fare`) vai de 0 até 512, com média bem acima da mediana. Tem valores bem extremos aí.

## 2. Tratar valores faltantes

In [7]:
df.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [8]:
(df.isna().mean() * 100).round(1)

,0
PassengerId,0.0
Survived,0.0
Pclass,0.0
Name,0.0
Sex,0.0
Age,19.9
SibSp,0.0
Parch,0.0
Ticket,0.0
Fare,0.0


**Cabin**: falta em 77% das linhas. Não tem como preencher isso de forma confiável, então vou remover a coluna. Mas antes guardo uma informação que pode ser útil: se o passageiro tinha cabine registrada ou não.

In [9]:
df["tem_cabine"] = df["Cabin"].notna()
df.groupby("Pclass")["tem_cabine"].mean().round(2)

,tem_cabine
Pclass,
1,0.81
2,0.09
3,0.02


Faz sentido: quase todo mundo da 1ª classe tem cabine registrada, enquanto na 3ª quase ninguém tem. Então o dado não está faltando por acaso, ele depende da classe.

In [10]:
df = df.drop(columns="Cabin")

**Embarked**: são só 2 linhas.

In [11]:
df[df["Embarked"].isna()]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,tem_cabine
61,62,1,1,"Icard, Miss. Amelie",female,38.0,0,0,113572,80.0,NaN,True
829,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.0,NaN,True


As duas viajaram no mesmo bilhete, na 1ª classe. Como são poucas, preencho com o porto mais comum (Southampton).

In [12]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

**Age**: faltam 177 idades (~20%). Remover essas linhas seria perder muita coisa, e preencher tudo com a média geral iria distorcer. A idade muda bastante conforme classe e sexo, então preencho com a mediana de cada grupo.

Também crio uma coluna marcando quais idades foram preenchidas, pra não misturar dado real com dado estimado na hora de analisar idade.

In [13]:
df.groupby(["Pclass", "Sex"])["Age"].median()

Pclass  Sex   
1       female    35.0
        male      40.0
2       female    28.0
        male      30.0
3       female    21.5
        male      25.0
Name: Age, dtype: float64

In [14]:
df["idade_estimada"] = df["Age"].isna()
df["Age"] = df["Age"].fillna(df.groupby(["Pclass", "Sex"])["Age"].transform("median"))
df.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


## 3. Ajustar tipos de coluna

In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
df["Survived"] = df["Survived"].astype(bool)
df["Pclass"] = pd.Categorical(df["Pclass"], categories=[1, 2, 3], ordered=True)
df["Sex"] = df["Sex"].astype("category")
df["Embarked"] = df["Embarked"].map({"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"}).astype("category")
df.dtypes

,0
PassengerId,int64
Survived,bool
Pclass,category
Name,object
Sex,category
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


## 4. Perguntas

### Pergunta 1: mulheres sobreviveram mais que homens?

In [17]:
por_sexo = df.groupby("Sex", observed=True)["Survived"].agg(["count", "sum", "mean"])
por_sexo.columns = ["passageiros", "sobreviventes", "taxa"]
por_sexo["taxa"] = (por_sexo["taxa"] * 100).round(1)
por_sexo

,passageiros,sobreviventes,taxa
Sex,,,
female,314,233,74.2
male,577,109,18.9


Sim, e a diferença é enorme: cerca de **74% das mulheres** sobreviveram contra só **19% dos homens**. Mesmo sendo minoria no navio (314 de 891), as mulheres são a maioria dos sobreviventes.

### Pergunta 2: a classe fez diferença?

In [18]:
por_classe = df.groupby("Pclass", observed=True).agg(
    passageiros=("Survived", "count"),
    taxa=("Survived", "mean"),
    tarifa_media=("Fare", "mean"),
    tarifa_mediana=("Fare", "median"),
)
por_classe["taxa"] = (por_classe["taxa"] * 100).round(1)
por_classe.round(2)

,passageiros,taxa,tarifa_media,tarifa_mediana
Pclass,,,,
1,216,63.0,84.15,60.29
2,184,47.3,20.66,14.25
3,491,24.2,13.68,8.05


Fez. A taxa de sobrevivência cai a cada classe: ~63% na 1ª, ~47% na 2ª e ~24% na 3ª.

Na tarifa da 1ª classe a média (84) fica bem acima da mediana (60), porque algumas pessoas pagaram valores muito altos. Aqui a mediana representa melhor o passageiro "típico".

Mas será que isso é só efeito do sexo? Talvez a 3ª classe tivesse mais homens. Cruzando as duas coisas:

In [19]:
(pd.crosstab(df["Pclass"], df["Sex"], values=df["Survived"], aggfunc="mean") * 100).round(1)

Sex,female,male
Pclass,,
1,96.8,36.9
2,92.1,15.7
3,50.0,13.5


In [20]:
pd.crosstab(df["Pclass"], df["Sex"])

Sex,female,male
Pclass,,
1,94,122
2,76,108
3,144,347


A classe pesa mesmo separando por sexo. Entre as mulheres, quase todas da 1ª e 2ª classe sobreviveram (97% e 92%), mas na 3ª só metade. Entre os homens, os da 1ª classe sobreviveram mais que o dobro dos da 2ª e 3ª.

Claro que isso mostra associação, não prova sozinho o motivo. A localização das cabines, o acesso aos botes e outras coisas devem ter influenciado.

### Pergunta 3: crianças tiveram mais chance?

Aqui uso só as idades reais, sem as que eu estimei, pra não enviesar o resultado.

In [21]:
idades_reais = df[~df["idade_estimada"]].copy()
idades_reais["faixa"] = pd.cut(
    idades_reais["Age"],
    bins=[0, 12, 18, 30, 50, 80],
    labels=["0-12", "13-18", "19-30", "31-50", "51-80"],
)
por_faixa = idades_reais.groupby("faixa", observed=True)["Survived"].agg(["count", "mean"])
por_faixa.columns = ["passageiros", "taxa"]
por_faixa["taxa"] = (por_faixa["taxa"] * 100).round(1)
por_faixa

,passageiros,taxa
faixa,,
0-12,69,58.0
13-18,70,42.9
19-30,270,35.6
31-50,241,42.3
51-80,64,34.4


In [22]:
criancas = idades_reais[idades_reais["Age"] <= 12]
(criancas.groupby("Pclass", observed=True)["Survived"].agg(["count", "mean"]).round(2))

,count,mean
Pclass,,
1,4,0.75
2,17,1.00
3,48,0.42


Crianças de até 12 anos tiveram a maior taxa de sobrevivência (~58%), acima de todas as outras faixas. Só que não foi igual pra todas: as poucas crianças da 1ª e 2ª classe quase todas sobreviveram, enquanto as da 3ª classe ficaram bem abaixo disso.

## 5. Conclusões

- O dataset tinha três colunas com valores faltantes. Removi `Cabin` (77% vazia), preenchi `Embarked` com o porto mais comum e `Age` com a mediana por classe e sexo, marcando as idades estimadas.
- Ajustei os tipos: `Survived` virou booleano e `Pclass`, `Sex` e `Embarked` viraram categorias.
- **Sexo** foi o fator mais forte: 74% das mulheres sobreviveram, contra 19% dos homens.
- **Classe** também pesou bastante, mesmo comparando pessoas do mesmo sexo.
- **Crianças** sobreviveram mais que adultos, mas a classe também fez diferença entre elas.
- No geral, a ideia de "mulheres e crianças primeiro" aparece nos dados, mas valeu bem mais pra quem estava na 1ª e 2ª classe.

Próximo passo: transformar esses resultados em gráficos na aula de matplotlib.